# ♻️ Notebook 4: Counting and Scalable Bloom Filters

Our bloom filter from notebook 1 has two painful limits:

1. **You cannot delete items.** Unsetting a bit could wipe out another item that happens to share it. This matters for things like a "logged-in users" filter that needs to drop people when they sign out.
2. **You have to know `n` up front.** Go past capacity and the false-positive rate degrades fast.

Two classic extensions fix these:

| Variant | Fixes | Cost |
|---|---|---|
| 🟩 **Counting bloom filter** | supports *delete* | ~4× more memory (counters, not bits) |
| 🟦 **Scalable bloom filter** | grows on demand | slightly higher FPR per lookup |

## Learning objectives
- Build a counting bloom filter and use `remove()`.
- Understand counter saturation (and why deletion still isn't perfectly safe).
- Build a scalable bloom filter that chains sub-filters as it fills.
- Know when each variant is the right tool.

## 🛠️ Setup

Pure Python. Run `uv sync` in this lab folder and select the `.venv` kernel in VS Code (top-right of the notebook). If the kernel doesn't appear: `Cmd+Shift+P` → **Reload Window**.

## 🟥 The problem: plain bloom filters can't delete

Let's prove it. Below, we add two items `"alice"` and `"bob"`, then try to "delete" alice by unsetting her bits. If bob happened to share any of those bits, a lookup for bob now returns **False** — a **false negative**, which is supposed to be impossible.

In [ ]:
import hashlib

class NaiveBloomFilter:
    def __init__(self, m, k):
        self.m, self.k = m, k
        self.bits = bytearray((m + 7) // 8)
    def _set(self, i):   self.bits[i // 8] |=  (1 << (i % 8))
    def _unset(self, i): self.bits[i // 8] &= ~(1 << (i % 8))
    def _get(self, i):   return (self.bits[i // 8] >> (i % 8)) & 1
    def _pos(self, x):
        d = hashlib.sha256(x.encode()).digest()
        h1 = int.from_bytes(d[:8], 'big'); h2 = int.from_bytes(d[8:16], 'big')
        for i in range(self.k):
            yield (h1 + i * h2) % self.m
    def add(self, x):
        for p in self._pos(x): self._set(p)
    def unsafe_remove(self, x):
        for p in self._pos(x): self._unset(p)  # ❌ dangerous!
    def __contains__(self, x):
        return all(self._get(p) for p in self._pos(x))

# Find two names whose hash positions overlap (guaranteed in a small filter)
def find_colliding_pair(m, k):
    ref = NaiveBloomFilter(m, k)
    ref.add('alice')
    for i in range(10_000):
        candidate = f'user_{i}'
        probe = NaiveBloomFilter(m, k); probe.add(candidate)
        # Overlap iff any bit set in both
        if any(a & b for a, b in zip(ref.bits, probe.bits)):
            return 'alice', candidate
    raise RuntimeError('no collision found')

a, b = find_colliding_pair(m=32, k=3)
print(f'Using names: {a!r} and {b!r} (their hash positions overlap)')

bf = NaiveBloomFilter(m=32, k=3)
bf.add(a); bf.add(b)
assert a in bf and b in bf          # both present before we touch anything
print(f'before remove:  {a} in bf = {a in bf}, {b} in bf = {b in bf}')

bf.unsafe_remove(a)
print(f'after  remove:  {a} in bf = {a in bf}, {b} in bf = {b in bf}')

# Assert the *bug*, don't just narrate it: b was added and is now reported absent.
assert b not in bf, "expected a false negative here — the demo is not proving its point"
print(f'\n❌ {b} was added but is now reported absent — a FALSE NEGATIVE.')
print('   Plain bloom filters simply cannot support delete.')

## 🟩 Counting bloom filter: bits → small counters

The fix is simple and elegant: instead of a bit at each position, store a small **counter** (usually 4 bits, so values 0–15).

- `add(x)` → **increment** each of the `k` counters by 1.
- `remove(x)` → **decrement** each of the `k` counters by 1.
- `x in bf` → all `k` counters are `> 0`.

Removing alice only decrements counters; if bob also touched those same cells, they stay `≥ 1` and bob is still reported as present. ✅

**Watch out:** if a counter is hit more than 15 times it would overflow. In practice you cap it at 15 (**saturation**) — once a counter is saturated you can't safely decrement it anymore, because you've lost count. This is why counting filters aren't a silver bullet.

In [ ]:
class CountingBloomFilter:
    """Counting bloom filter with 4-bit counters packed 2-per-byte."""
    COUNTER_MAX = 15  # 4-bit counters

    def __init__(self, m, k):
        self.m, self.k = m, k
        # Two 4-bit counters per byte → ceil(m/2) bytes
        self.counters = bytearray((m + 1) // 2)
        self.saturated = 0  # how many times we hit the cap

    def _get(self, i):
        byte = self.counters[i // 2]
        return (byte >> (4 * (i % 2))) & 0xF

    def _set(self, i, value):
        value &= 0xF
        idx = i // 2
        shift = 4 * (i % 2)
        self.counters[idx] = (self.counters[idx] & ~(0xF << shift)) | (value << shift)

    def _pos(self, x):
        d = hashlib.sha256(x.encode()).digest()
        h1 = int.from_bytes(d[:8], 'big'); h2 = int.from_bytes(d[8:16], 'big')
        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    def add(self, x):
        for p in self._pos(x):
            c = self._get(p)
            if c < self.COUNTER_MAX:
                self._set(p, c + 1)
            else:
                self.saturated += 1  # stay at max — can no longer delete safely

    def remove(self, x):
        if x not in self:
            return False  # safety: don't decrement for items we never added
        for p in self._pos(x):
            c = self._get(p)
            if 0 < c < self.COUNTER_MAX:
                self._set(p, c - 1)
            # if c == COUNTER_MAX we leave it — it's saturated
        return True

    def __contains__(self, x):
        return all(self._get(p) > 0 for p in self._pos(x))

    def memory_bytes(self):
        return len(self.counters)

print('CountingBloomFilter ready ✅')

### Demo: add, remove, and verify no false negatives on the survivors

In [ ]:
cbf = CountingBloomFilter(m=2048, k=5)

# A tiny "who is currently logged in" set
logged_in = ['alice', 'bob', 'carol', 'dan', 'eve']
for u in logged_in:
    cbf.add(u)
assert all(u in cbf for u in logged_in)
print('all logged in:', {u: (u in cbf) for u in logged_in})

# alice and dan log out
assert cbf.remove('alice') is True
assert cbf.remove('dan') is True

survivors = ['bob', 'carol', 'eve']
print('\nafter logout:', {u: (u in cbf) for u in logged_in})

# The survivors MUST still be present — that is the whole point of counters.
assert all(u in cbf for u in survivors), 'counting filter produced a false negative'
assert 'alice' not in cbf and 'dan' not in cbf
print('\nbob, carol, eve survive the removals — no false negatives.')

# remove() must never take a counter below zero, and must refuse unknown items.
assert cbf.remove('mallory') is False, 'removing an item we never added must be a no-op'
assert all(cbf._get(i) >= 0 for i in range(cbf.m)), 'counter underflow'
print("remove('mallory') refused — an unknown item cannot decrement anything.")

### ⚠️ The hazard counters *don't* fix

`remove()` guards against the obvious mistake (decrementing for an item that isn't
there) — but the membership test it uses is itself probabilistic. If an item you never
added happens to be a **false positive**, `remove()` believes it *is* a member and
decrements those `k` counters. Any real member sharing one of those cells can drop to
zero, and now the filter has a **false negative** — the thing counting filters were
supposed to make impossible.

This is not theoretical; it is the reason production systems only delete keys they have
independently confirmed exist. Let's force it with a deliberately over-stuffed filter.

In [ ]:
# Tiny filter, lots of members => plenty of false positives to trip over.
tiny = CountingBloomFilter(m=64, k=3)
members = [f'user-{i}' for i in range(30)]
for u in members:
    tiny.add(u)

# Find a non-member that the filter wrongly believes it contains.
ghost = next(g for g in (f'ghost-{i}' for i in range(10_000)) if g in tiny)
print(f'{ghost!r} was never added, but the filter says it is present (false positive).')

tiny.remove(ghost)                      # remove() trusts the membership test...
broken = [u for u in members if u not in tiny]
assert broken, 'expected removing a false positive to damage a real member'
print(f'After removing it, {len(broken)} REAL member(s) vanished: {broken[:3]}')
print('\n❌ A counting bloom filter is only safe if you delete items you know exist.')

### Memory cost

Counting bloom filters use **4 bits per counter** instead of 1 bit — so they're about **4× the memory** of a plain bloom filter with the same `m`. That's the price of `remove()`.

In [ ]:
plain_bytes = (cbf.m + 7) // 8
counting_bytes = cbf.memory_bytes()
print(f'same m={cbf.m:,} positions:')
print(f'  plain  bloom filter : {plain_bytes:>5} bytes')
print(f'  counting bloom filt.: {counting_bytes:>5} bytes  ({counting_bytes/plain_bytes:.1f}× larger)')

## 🟦 Scalable bloom filter: grow as needed

What if you don't know `n` in advance? A scalable bloom filter (Almeida et al., 2007) is a **list of bloom filters**. When the current filter fills up, start a new one with **more bits** and a **tighter FPR** so the *compound* FPR stays bounded.

- `add(x)` → add to the newest sub-filter; if it's full, allocate a new bigger one.
- `x in sbf` → **any** sub-filter says yes.

**Why tighter FPR each time?** A union-bound: if each of `L` sub-filters has FPR `p_i`, the overall FPR is roughly `sum(p_i)`. Shrinking each new `p_i` by a fixed ratio `r` (commonly `0.5`) keeps the sum from exploding: `p * (1 + r + r² + …) ≤ p / (1 - r)`.

In [ ]:
import math

class _SubFilter:
    """Plain bloom filter that also tracks how full it is."""
    def __init__(self, capacity, fpr):
        self.capacity = capacity
        self.count = 0
        self.m = int(math.ceil(-capacity * math.log(fpr) / (math.log(2) ** 2)))
        self.k = max(1, int(round((self.m / capacity) * math.log(2))))
        self.bits = bytearray((self.m + 7) // 8)
    def _set(self, i): self.bits[i // 8] |= 1 << (i % 8)
    def _get(self, i): return (self.bits[i // 8] >> (i % 8)) & 1
    def _pos(self, x):
        d = hashlib.sha256(x.encode()).digest()
        h1 = int.from_bytes(d[:8], 'big'); h2 = int.from_bytes(d[8:16], 'big')
        for i in range(self.k): yield (h1 + i * h2) % self.m
    def add(self, x):
        for p in self._pos(x): self._set(p)
        self.count += 1
    def __contains__(self, x):
        return all(self._get(p) for p in self._pos(x))
    def is_full(self):
        return self.count >= self.capacity

class ScalableBloomFilter:
    """Grows by adding new, bigger sub-filters as it fills."""
    def __init__(self, initial_capacity=1000, target_fpr=0.01,
                 growth=2, tightening=0.5):
        # growth     : next sub-filter holds `growth`× more items
        # tightening : next sub-filter uses `tightening`× the FPR budget
        self.growth = growth
        self.tightening = tightening
        self._target_fpr = target_fpr
        self._initial_capacity = initial_capacity
        self.filters = [_SubFilter(initial_capacity, target_fpr * (1 - tightening))]

    def add(self, x):
        current = self.filters[-1]
        if current.is_full():
            # allocate a new, bigger, tighter sub-filter
            new_cap = current.capacity * self.growth
            new_fpr = self._target_fpr * (1 - self.tightening) * (self.tightening ** len(self.filters))
            current = _SubFilter(new_cap, new_fpr)
            self.filters.append(current)
        current.add(x)

    def __contains__(self, x):
        return any(x in f for f in self.filters)

    def stats(self):
        total_bits = sum(f.m for f in self.filters)
        total_items = sum(f.count for f in self.filters)
        return {
            'sub_filters': len(self.filters),
            'total_items': total_items,
            'total_bits': total_bits,
            'total_bytes': (total_bits + 7) // 8,
        }

print('ScalableBloomFilter ready ✅')

### Demo: insert 10× more than we planned

We'll configure for `initial_capacity=1000` but insert **50,000 items**. Watch how the filter spawns more sub-filters on the fly, while the measured FPR stays close to the 1% budget.

In [ ]:
sbf = ScalableBloomFilter(initial_capacity=1000, target_fpr=0.01)

inserted = [f'item-{i}' for i in range(50_000)]
for x in inserted:
    sbf.add(x)

# No false negatives: a chain of filters is still a union of filters.
assert all(x in sbf for x in inserted), 'scalable filter lost an item'
print(f'False negatives: 0 / {len(inserted):,}')

# Measure FPR on fresh probes and check it against the compound budget.
# Sub-filter i is built for p_i = p*(1-r)*r^i, and sum(p_i) = p — so the chain
# stays *near* the 1% target no matter how many sub-filters it grew. It can land a
# hair over: each sub-filter rounds k to an integer, which costs a little accuracy.
probes = [f'probe-{i}' for i in range(20_000)]
fp = sum(1 for q in probes if q in sbf)
fpr = fp / len(probes)
print(f'False positives: {fp}/{len(probes)} ({fpr:.2%})  (budget: ~1.00%)')
assert fpr <= 0.012, f'compound FPR {fpr:.2%} blew the budget'

print('\nfilter stats:', sbf.stats())
print(f'sub-filter capacities: {[f.capacity for f in sbf.filters]}')
# Capacities must grow geometrically, and the chain must have actually grown.
assert len(sbf.filters) > 5
assert [f.capacity for f in sbf.filters] == [1000 * 2 ** i for i in range(len(sbf.filters))]

### Tradeoff to notice

- ✅ No more "you must know `n` upfront." Just keep inserting.
- ❌ Each lookup now queries **every** sub-filter in the chain, so reads get slower as the filter grows.
- ❌ Still no deletion. Combine with counting bloom filters if you need both (at even more memory).

## 🌍 Where you'll actually see these

- **Counting bloom filters**: routing tables in networking hardware, deletable caches, 
  privacy-preserving ad counting (e.g., RAPPOR).
- **Scalable bloom filters**: long-lived streams where `n` is unknown — deduplication in event pipelines, **Redis 4+'s `BF.ADD`** (via the RedisBloom module) uses this pattern, some anti-abuse systems, the `pybloom-live` library.
- **Neither**: if you need exact membership, a **Cuckoo filter** supports deletion, is usually smaller, and returns items. Worth looking up as the modern successor.

## 🎓 Takeaways

1. If you need **delete** → counting bloom filter (≈4× memory, watch counter saturation).
2. If you don't know **n** upfront → scalable bloom filter (slower reads as it grows).
3. If you need **both**, or to **list items** → probably a different data structure (Cuckoo filter, quotient filter, or just a hash set + LRU).